# Greenville, South Carolina — City Levy, 4:1 Split-Rate LVT

This notebook models a revenue-neutral shift of the **City of Greenville's own property-tax levy**
(81.4 mills) from the current uniform tax on land + buildings to a **4:1 split-rate**, where land
is taxed at 4× the rate on buildings. County, school-district, and special-district levies stay as
they are and are not modeled.

**Scope decisions (agreed up front)**

| Decision | Choice |
|---|---|
| Geography | Parcels whose representative point falls inside the official City of Greenville boundary (Greenville County GIS, November 2026) |
| Levy | City of Greenville levy only: 81.4 mills (Greenville County Auditor 2026 millage sheet; unchanged since at least 2021) |
| Reform | **4:1 split-rate plus a flat $50,000 building exemption on every taxable parcel**, revenue-neutral on the modeled taxable base. The 65+ homestead exemption is allocated pro rata between land and building |
| SC assessment ratios | Kept: 4% for owner-occupied legal residences, 6% for all other locally assessed real property. The ratio applies to **both** land and buildings, so only the millage is split |
| Existing relief | Kept: legal-residence 4% ratio, the 65+/disabled/blind homestead exemption, full exemptions (government, church, charity, hospital, etc.), and the Act 388 15% reassessment cap (the county's *taxable* market value is used, not fair market value) |
| Homestead | **$75,000**, the tax-year-2026 amount under the state budget proviso (pending S.768 would make an increase permanent); eligibility 65+, totally disabled or legally blind, on a legal residence. The baseline and the LVT scenario both use $75K; Section 6 also reports the change from the $50K permanent statute |
| Data vintage | **Tax year 2026** bills on values from the **2025 countywide reassessment** (county GIS export, 2026-11-23) |
| FILOT / special arrangements | Held out of the shift; TIF parcels are modeled like any other parcel |

## Section 1 — Imports, constants, `.env`

In [1]:
import sys
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

# Constants
CITY_NAME = 'greenville_sc'
STATE_FIPS = '45'
COUNTY_FIPS = '045'
MODEL_TYPE = 'split_rate_4to1_bldg_exempt_50k'
LAND_IMPROVEMENT_RATIO = 4.0

TAX_YEAR = 2026                 # bills in the county export; values from the 2025 reassessment
CITY_MILLAGE = 81.4             # City of Greenville levy, Greenville County 2026 millage sheet (districts 500-517)
DISTRICT_500_TOTAL_MILLAGE = 330.3  # full stack for district 500 in 2026 (county 59.9 + school 188.7 + city 81.4 + arena 0.3)
RATIO_LEGAL_RESIDENCE = 0.04    # S.C. Code 12-43-220(c)
RATIO_OTHER = 0.06              # S.C. Code 12-43-220(e)
HOMESTEAD_CURRENT = 50_000      # S.C. Code 12-37-250 permanent amount (65+/disabled/blind)
HOMESTEAD_PROPOSED = 75_000     # tax-year-2026 amount (budget proviso 117.220), same eligibility
HOMESTEAD_MODELED = HOMESTEAD_PROPOSED
BUILDING_EXEMPTION = 50_000     # first $50K of building market value untaxed on every taxable parcel
HOLDOUT_BILL_SHARE = 0.5        # observed bill < 50% of the statutory bill => exempt / FILOT / special arrangement

# Parcel-map export identity columns
PARCEL_ID_COL = 'PIN'
OWNER_NAME_COL = None          # 2021 owner names are not current; not exported
OWNER_ADDRESS_COL = 'site_address'
PARCEL_URL_TEMPLATE = None      # county property-record portal is session-based; no stable deep link verified

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Load parcel data

**Source.** A file geodatabase exported from Greenville County GIS on 2026-11-23
(`LVTShiftGreenvilleSC20261123.gdb`, provided by the Center for Land Economics), cached in `data/`.
It holds:

| Layer | Use |
|---|---|
| `TaxParcelLabel20261123` (244,930 parcels) | Values and bills: `LANDVAL`, `BLDGVAL`, `TAXMKTVAL` (capped taxable market value = land + building), `FAIRMKTVAL` (uncapped), `TOTTAX` (the full observed bill), `LANDUSE`, `PROPTYPE` |
| `MunicipalBoundary20261123` | Official city limits (`NAME = 'GREENVILLE'`, 31.17 sq mi, edited 2026-11-19) |
| `TaxDistrict20261123` | Tax-district polygons (`TDISTNUM`); the 500-series districts are the City of Greenville |
| `ParkingLot20261123` | Surface parking polygons (not used in this model) |

**Tax year.** The values come from the **2025 countywide reassessment**, and `TOTTAX` is the
**tax-year 2026** bill. In district 046 (1,613 valued parcels) the bills imply 290.4 mills at the 6%
ratio, exactly the 2026 sheet's 290.4 (2025: 286.5).

**What the parcel layer lacks.** It has no parcel number, owner, tax district, assessment-ratio code
or homestead flag. Tax district comes from the district polygons. Assessment class and homestead
status are inferred from each parcel's 2026 bill (Section 3a). Parcel number, owner name, legal
description and site address are carried over **by location** from the county's tax-year-2021 parcel
layer (`county_parcels_raw.gpq`, an ArcGIS Online copy of the county `TaxParcel` class). They are used
only to recognize government, nonprofit, HOA and utility owners. Ownership
changes since 2021 are not reflected, so owner names are not exported to the parcel map.

**City filter.** A parcel is in the city if its representative point falls inside the official
municipal boundary.

**Reassessment cap and the land/building split.** After the 2025 reassessment, 72% of city parcels
are limited by the 15% cap, so their taxable value (`TAXMKTVAL`) is below fair market value. The
county takes the cap off the building first: `LANDVAL` stays at **full, uncapped** land value and
`BLDGVAL = TAXMKTVAL − LANDVAL`, falling to zero only when the cap exceeds the building (on vacant
lots `LANDVAL` simply equals the capped total).
On capped single-family homes, land is 25.8% of fair market value (uncapped homes: 26.6%), but
39.6% of taxable value. Booked that way, the entire cap discount lands on the building, which
understates building value and overstates land on most parcels. The model re-splits each parcel's
taxable value in the proportions of its uncapped land and building values:

```
taxable_land     = TAXMKTVAL × LANDVAL / FAIRMKTVAL
taxable_building = TAXMKTVAL − taxable_land
```

The cap stays exactly as large; it is shared by land and building in proportion to their value.
Parcels with no booked building value are left as all land.

In [2]:
# Step 2a: county parcels, official boundary and tax districts (TY2026 export)
GDB_PATH = DATA_DIR / 'LVTShiftGreenvilleSC20261123.gdb'
county = gpd.read_file(GDB_PATH, layer='TaxParcelLabel20261123')
CRS_SC = county.crs
muni = gpd.read_file(GDB_PATH, layer='MunicipalBoundary20261123').set_crs(CRS_SC, allow_override=True)
tax_districts = gpd.read_file(GDB_PATH, layer='TaxDistrict20261123').set_crs(CRS_SC, allow_override=True)
city_boundary = muni[muni['NAME'].str.upper() == 'GREENVILLE'].dissolve()
print(f"County parcels: {len(county):,}   city boundary: {city_boundary.area.sum() / 27_878_400:.2f} sq mi")

# Step 2b: keep parcels whose representative point is inside the official city boundary
_pts = county[['geometry']].copy()
_pts['geometry'] = county.representative_point()
_inside = gpd.sjoin(_pts, city_boundary[['geometry']], predicate='within').index.unique()
gdf = county.loc[_inside].copy()
print(f"City parcels (official boundary): {len(gdf):,}")

# Step 2c: tax district from the district polygons
_pts = gdf[['geometry']].copy()
_pts['geometry'] = gdf.representative_point()
_dist = gpd.sjoin(_pts, tax_districts[['TDISTNUM', 'geometry']], predicate='within')
_dist = _dist[~_dist.index.duplicated()]
gdf['DIST'] = _dist['TDISTNUM'].reindex(gdf.index).fillna('').astype(str).str.strip()
CITY_DISTRICTS = sorted(d for d in gdf['DIST'].unique() if d.startswith('5'))
gdf['in_city_tax_district'] = gdf['DIST'].isin(CITY_DISTRICTS)
print("City tax districts present:", CITY_DISTRICTS)
print(f"Parcels inside the boundary but outside a city tax district: {(~gdf['in_city_tax_district']).sum():,}",
      gdf.loc[~gdf['in_city_tax_district'], 'DIST'].value_counts().head(8).to_dict())

# Step 2d: carry identity fields over from the TY2021 parcel layer by location
LEGACY_PATH = DATA_DIR / 'county_parcels_raw.gpq'
if not LEGACY_PATH.exists():
    _legacy = get_feature_data_with_geometry(
        dataset_name='TaxParcel02172021_gdb',
        base_url='https://services3.arcgis.com/bjOyhhlaaCIcmO3H/arcgis/rest/services',
        layer_id=0, paginate=True, verbose=False)
    _legacy.to_parquet(LEGACY_PATH)
LEGACY_COLS = ['PIN', 'OWNAM1', 'DESCR', 'SUBDIV', 'STRNUM', 'LOCATE', 'RRETCD', 'HMSTCD']
legacy = gpd.read_parquet(LEGACY_PATH, columns=LEGACY_COLS + ['geometry']).to_crs(CRS_SC)
_lj = gpd.sjoin(_pts, legacy, predicate='within')
_lj = _lj[~_lj.index.duplicated()]
for _c in LEGACY_COLS:
    gdf[_c] = _lj[_c].reindex(gdf.index)
gdf['RRETCD_2021'] = gdf.pop('RRETCD').fillna('').astype(str).str.strip()
gdf['HMSTCD_2021'] = gdf.pop('HMSTCD').fillna('').astype(str).str.strip()
print(f"Matched to a 2021 parcel: {gdf['PIN'].notna().mean():.1%}")

# Parcel id: the 2021 PIN where it is unique, otherwise a stable new id
_dup = gdf['PIN'].duplicated(keep=False) | gdf['PIN'].isna()
gdf['PIN'] = np.where(_dup, 'GVL26-' + gdf.index.astype(str), gdf['PIN'])

County parcels: 244,930   city boundary: 31.89 sq mi


City parcels (official boundary): 28,325
City tax districts present: ['500', '501', '502', '511', '512', '515', '516', '517']
Parcels inside the boundary but outside a city tax district: 0 {}


Matched to a 2021 parcel: 99.6%


In [3]:
# Step 2e: column checks
for _c in ['LANDVAL', 'BLDGVAL', 'TAXMKTVAL', 'FAIRMKTVAL', 'TOTTAX', 'GIS_ACRES']:
    gdf[_c] = pd.to_numeric(gdf[_c], errors='coerce').fillna(0).clip(lower=0)
gdf['LANDUSE'] = gdf['LANDUSE'].fillna('').astype(str).str.strip()
gdf['site_address'] = (gdf['STRNUM'].fillna('').astype(str).str.strip() + ' '
                       + gdf['LOCATE'].fillna('').astype(str).str.strip()).str.strip()

# The county books LANDVAL at full (uncapped) market value and BLDGVAL = TAXMKTVAL − LANDVAL, so the
# whole 15% reassessment-cap discount lands on the building. Re-split the capped taxable value in
# proportion to the uncapped land/building mix (see markdown above).
gdf['LANDVAL_RAW'] = gdf['LANDVAL']
gdf['BLDGVAL_RAW'] = gdf['BLDGVAL']
# Only parcels with booked building value carry full land value in LANDVAL; where the cap used up the
# building (BLDGVAL = 0, e.g. vacant lots) LANDVAL already equals the capped total, so they stay all land.
_has_bldg = gdf['BLDGVAL_RAW'] > 0
_land_share_fmv = np.where(_has_bldg & (gdf['FAIRMKTVAL'] > 0), (gdf['LANDVAL_RAW'] / gdf['FAIRMKTVAL']).clip(0, 1),
                           np.where(gdf['TAXMKTVAL'] > 0, gdf['LANDVAL_RAW'] / gdf['TAXMKTVAL'], 0))
gdf['LANDVAL'] = (gdf['TAXMKTVAL'] * _land_share_fmv).round(0)
gdf['BLDGVAL'] = (gdf['TAXMKTVAL'] - gdf['LANDVAL']).clip(lower=0)
_capped = gdf['TAXMKTVAL'] < gdf['FAIRMKTVAL'] - 5
_sfr_chk = (gdf['LANDUSE'] == '1100') & (gdf['TAXMKTVAL'] > 50_000)
print(f"Single-family land share of taxable value, as booked: capped {(gdf['LANDVAL_RAW'] / gdf['TAXMKTVAL'])[_sfr_chk & _capped].median():.3f} "
      f"vs uncapped {(gdf['LANDVAL_RAW'] / gdf['TAXMKTVAL'])[_sfr_chk & ~_capped].median():.3f}; "
      f"after proportional re-split: capped {(gdf['LANDVAL'] / gdf['TAXMKTVAL'])[_sfr_chk & _capped].median():.3f}")

print(f"LANDVAL_RAW + BLDGVAL_RAW == TAXMKTVAL: {np.isclose(gdf['LANDVAL_RAW'] + gdf['BLDGVAL_RAW'], gdf['TAXMKTVAL'], atol=5).mean():.1%}")
print(f"Capped (TAXMKTVAL < FAIRMKTVAL): {(gdf['TAXMKTVAL'] < gdf['FAIRMKTVAL'] - 5).mean():.1%}")
print(f"$0 land: {(gdf['LANDVAL'] == 0).sum():,}   $0 building: {(gdf['BLDGVAL'] == 0).sum():,}   $0 total: {(gdf['TAXMKTVAL'] == 0).sum():,}")
print(f"Median land ${gdf['LANDVAL'].median():,.0f}   median building ${gdf['BLDGVAL'].median():,.0f}")
print(gdf[['LANDVAL', 'BLDGVAL', 'TAXMKTVAL', 'FAIRMKTVAL', 'TOTTAX']].describe().round(0).T)

Single-family land share of taxable value, as booked: capped 0.396 vs uncapped 0.266; after proportional re-split: capped 0.258
LANDVAL_RAW + BLDGVAL_RAW == TAXMKTVAL: 100.0%
Capped (TAXMKTVAL < FAIRMKTVAL): 72.5%
$0 land: 191   $0 building: 4,670   $0 total: 164
Median land $82,482   median building $190,000
              count      mean        std  min       25%       50%       75%          max
LANDVAL     28325.0  184057.0   717614.0  0.0   35220.0   82482.0  169818.0   48116380.0
BLDGVAL     28325.0  458143.0  2249881.0  0.0   71793.0  190000.0  378606.0   86911140.0
TAXMKTVAL   28325.0  642200.0  2668090.0  0.0  128040.0  294000.0  555590.0   91225290.0
FAIRMKTVAL  28325.0  912552.0  3712160.0  0.0  212570.0  381630.0  706570.0  183592000.0
TOTTAX      28325.0    7891.0    47583.0  0.0    1083.0    2704.0    5141.0    1831730.0


## Section 3 — Classify parcels and validate

### 3a. Assessment class and homestead, read from the 2026 bill

South Carolina assesses locally assessed real property at **4%** of (capped) market value for an
owner-occupied legal residence and **6%** for everything else (rentals, second homes, commercial,
vacant land). Legal residences also skip school operating millage (Act 388). In district 500 a
6% parcel pays about **20.4 per $1,000** of market value (≈340 mills) and a legal residence about
**7.8** (≈194 mills), so the bill reveals the class.

For each parcel the observed `TOTTAX` is compared with three candidate bills at its district's
estimated full-stack millage:

```
6% class:                     TAXMKTVAL × 6% × mills_6
legal residence:              TAXMKTVAL × 4% × mills_4
legal residence + homestead:  (TAXMKTVAL − $75,000) × 4% × mills_4
```

The closest candidate within ±20% sets the class. `mills_6` and `mills_4` are estimated per district
from parcels whose bills sit in each class's band. For tax year 2026 the homestead exemption for
owners 65+, totally disabled or legally blind is **$75,000**, under a one-year state budget proviso
for owners already eligible in 2025; the permanent statute is $50,000. Parcels whose bill matches
no candidate are sorted by level: a residential bill below the 4%-class level (under $9.50 per
$1,000) is a legal residence, usually a recent sale whose bill was computed on the earlier capped
value while the export already shows the post-sale value; a bill above that level is 6% class;
parcels with essentially no bill (exempt, fee-in-lieu, new plats) use their 2021 ratio and
homestead codes.

> Note on manufacturing: 10.5% applies to manufacturing and utility property, which the SC
> Department of Revenue assesses. Those values are not in the county parcel layer.

In [4]:
# Step 3a: infer class from the bill
_r = np.where(gdf['TAXMKTVAL'] > 0, gdf['TOTTAX'] / gdf['TAXMKTVAL'] * 1000, np.nan)   # $ per $1,000 market value
_m6 = pd.Series(_r / 0.06, index=gdf.index)
_m4 = pd.Series(_r / 0.04, index=gdf.index)
_valued = gdf['TAXMKTVAL'] > 30_000
_band6 = _valued & _m6.between(240, 520)
_band4 = _valued & _m4.between(120, 260) & ~_band6
mills_6 = _m6[_band6].groupby(gdf['DIST']).median()
mills_4 = _m4[_band4].groupby(gdf['DIST']).median()
_n6 = _band6.groupby(gdf['DIST']).sum()
_n4 = _band4.groupby(gdf['DIST']).sum()
mills_6 = mills_6[_n6.reindex(mills_6.index) >= 20]
mills_4 = mills_4[_n4.reindex(mills_4.index) >= 20]
gdf['mills_6'] = gdf['DIST'].map(mills_6).fillna(_m6[_band6].median())
gdf['mills_4'] = gdf['DIST'].map(mills_4).fillna(_m4[_band4].median())
print("Estimated full-stack mills, city districts (6% class / legal residence):")
print(pd.DataFrame({'mills_6': mills_6, 'mills_4': mills_4}).reindex(CITY_DISTRICTS).round(1))

_cands = pd.DataFrame({
    '6%': gdf['TAXMKTVAL'] * RATIO_OTHER * gdf['mills_6'] / 1000,
    'LR': gdf['TAXMKTVAL'] * RATIO_LEGAL_RESIDENCE * gdf['mills_4'] / 1000,
    'LR+HS': (gdf['TAXMKTVAL'] - HOMESTEAD_PROPOSED).clip(lower=0) * RATIO_LEGAL_RESIDENCE * gdf['mills_4'] / 1000,
})
_err = np.abs(np.log(gdf['TOTTAX'].clip(lower=1).values[:, None] / _cands.clip(lower=1).values))
_best = _err.argmin(axis=1)
_ok = (_err.min(axis=1) <= np.log(1.2)) & (gdf['TOTTAX'] > 0).values & (gdf['TAXMKTVAL'] > 0).values
gdf['bill_class'] = np.where(_ok, np.array(_cands.columns)[_best], 'unresolved')

# Unresolved bills. A bill below the legal-residence level (under 9.5 per $1,000 of market value)
# can only come from the 4% class, typically a recent sale whose bill was computed on the earlier,
# capped value. A bill above it is 6% class. Parcels with essentially no bill use their 2021 codes.
_lr21 = gdf['RRETCD_2021'] == '2'
_hs21 = gdf['HMSTCD_2021'] == '*'
_rr = pd.Series(_r, index=gdf.index)
_un = gdf['bill_class'] == 'unresolved'
_resid_use = gdf['PROPTYPE'].fillna('').str.upper().isin(['RESIDENTIAL', 'MOBILE HOME']) | gdf['LANDUSE'].str.startswith('11')
_un_lr = _un & _resid_use & _rr.between(0.5, 9.5)
_un_6 = _un & (_rr >= 9.5)
_un_fb = _un & ~_un_lr & ~_un_6
gdf.loc[_un_lr, 'bill_class'] = 'unresolved → LR (bill below 4% level)'
gdf.loc[_un_6, 'bill_class'] = 'unresolved → 6% (bill above 4% level)'
gdf.loc[_un_fb, 'bill_class'] = 'unresolved → 2021 codes'
gdf['legal_residence'] = np.where(_un_fb, _lr21,
                                  gdf['bill_class'].isin(['LR', 'LR+HS']) | _un_lr).astype(int)
gdf['homestead_flag'] = np.where(_un, (gdf['legal_residence'] == 1) & _hs21,
                                 gdf['bill_class'] == 'LR+HS').astype(int)
gdf['assessment_ratio'] = np.where(gdf['legal_residence'] == 1, RATIO_LEGAL_RESIDENCE, RATIO_OTHER)
print(gdf['bill_class'].value_counts())
print(f"Legal residences: {gdf['legal_residence'].sum():,}   with homestead exemption: {gdf['homestead_flag'].sum():,}")
_m = gdf['bill_class'].isin(['LR', 'LR+HS', '6%']) & (gdf['RRETCD_2021'] != '')
print(f"Agreement with the 2021 ratio code where the 2026 bill resolves cleanly: "
      f"{((gdf.loc[_m, 'legal_residence'] == 1) == _lr21[_m]).mean():.1%} (differences are mostly ownership changes)")

Estimated full-stack mills, city districts (6% class / legal residence):
      mills_6  mills_4
DIST                  
500     340.5    193.4
501     334.1    191.9
502     335.0    191.3
511     334.9    191.7
512     342.1    193.5
515     335.2    192.6
516     341.8    193.4
517       NaN      NaN
bill_class
LR                                       9722
6%                                       8213
unresolved → LR (bill below 4% level)    3054
LR+HS                                    2713
unresolved → 2021 codes                  2481
unresolved → 6% (bill above 4% level)    2142
Name: count, dtype: int64
Legal residences: 15,634   with homestead exemption: 4,000
Agreement with the 2021 ratio code where the 2026 bill resolves cleanly: 83.5% (differences are mostly ownership changes)


### 3b. Exempt, FILOT and special-arrangement parcels (held out)

The parcel layer has no explicit exemption or FILOT flag. Exempt parcels (government, churches,
hospitals, universities) **keep their appraised values**, and their `TOTTAX` holds only small
non-ad-valorem charges. FILOT parcels pay a negotiated fee instead of millage. Both are identified
by comparing the **observed bill** with the **statutory bill** for the parcel's class:

```
statutory_bill = (TAXMKTVAL − homestead) × ratio × district_mills / 1000
low_bill       = TOTTAX < 50% × statutory_bill
```

A low bill on its own doesn't settle whether a parcel is exempt, FILOT, or new construction billed
for part of the year. Step 3c-2 sorts the low-bill parcels once the property categories exist.
Held-out parcels are excluded from the revenue-neutral solve, the export and the charts, as the
repo requires, and are reported by count.

In [5]:
# Step 3b: statutory vs observed bill
gdf['district_mills'] = np.where(gdf['legal_residence'] == 1, gdf['mills_4'], gdf['mills_6'])
_hs_mv = np.where(gdf['homestead_flag'] == 1, np.minimum(HOMESTEAD_PROPOSED, gdf['TAXMKTVAL']), 0.0)
gdf['statutory_bill'] = (gdf['TAXMKTVAL'] - _hs_mv).clip(lower=0) * gdf['assessment_ratio'] * gdf['district_mills'] / 1000
gdf['bill_share'] = np.where(gdf['statutory_bill'] > 0, gdf['TOTTAX'] / gdf['statutory_bill'], np.nan)
gdf['low_bill'] = ((gdf['statutory_bill'] >= 100)
                   & (gdf['TOTTAX'] < HOLDOUT_BILL_SHARE * gdf['statutory_bill'])).astype(int)
print(f"Parcels billed < {HOLDOUT_BILL_SHARE:.0%} of statutory: {gdf['low_bill'].sum():,} "
      "(classified into exempt / FILOT / bill-timing after the categories are built, Step 3c-2)")

Parcels billed < 50% of statutory: 3,242 (classified into exempt / FILOT / bill-timing after the categories are built, Step 3c-2)


### 3c. Property categories

The categories come from the county `LANDUSE` code, checked against the code list in the county's
*Physical Database Design* (January 2026): for example 240/250/270 are hotels, 370/371 parking
garages and lots, 610 fast food, 620 full-service restaurants, 810 churches and 821 government
property, with `PROPTYPE` as the fallback. The code list has no condominium code; condo units are
coded 1100 (single family) and drawn as footprint-sized slivers. A `1100` parcel under
**1,500 sq ft** of mapped area is classed **Condo & Townhome** (condo units median 489 sq ft;
houses median 9,549 sq ft). The cutoff also captures fee-simple townhouses on very small lots, hence
the combined label. Greenville condos carry real land value, so no St. Paul–style condo collapse is
needed.
Any parcel with $0 building value becomes **Vacant Land**, except the parking codes, where a
paved lot with no building is the expected record.

In [6]:
# Step 3c: categories
LANDUSE_MAP = {
    '1100': 'Single Family Residential',
    '1101': 'Other Residential',
    '1180': 'Vacant Land',
    '1181': 'Other',                         # residential common area
    '110': 'Small Multi-Family (2-4 units)',
    '112': 'Small Multi-Family (2-4 units)',
    '120': 'Large Multi-Family (5+ units)',
    '122': 'Large Multi-Family (5+ units)',
    '141': 'Other Commercial',               # assisted living / memory care
    '205': 'Other',                          # commercial common area
    '240': 'Hotel', '250': 'Hotel', '270': 'Hotel',
    '370': 'Transportation - Parking',       # parking decks
    '371': 'Transportation - Parking',       # surface lots
    '821': 'Other',                          # government (county code list)
    '6800': 'Vacant Land',                   # vacant commercial
    '770': 'Other Commercial',               # recreation / arena
    '810': 'Other',                          # churches
    '850': 'Other',                          # schools / institutional
    '851': 'Other Commercial',
    '872': 'Other Commercial',               # funeral homes
    '910': 'Other Commercial',               # self-storage
}
OFFICE_CODES = {'409', '410', '411', '420', '421', '423', '425', '430', '431', '720'}
RETAIL_PREFIXES = ('3', '5', '6')            # 300-360 auto/gas/car wash, 510-590 retail, 610-632 restaurants


def classify(row) -> str:
    code = str(row['LANDUSE']).strip()
    if code == '1100':
        if row['_parcel_sqft'] < CONDO_MAX_SQFT:
            return 'Condo & Townhome'
        return 'Single Family Residential'
    if code in LANDUSE_MAP:
        return LANDUSE_MAP[code]
    if code in OFFICE_CODES:
        return 'Office / Commercial Condo'
    if len(code) == 3 and code.startswith(RETAIL_PREFIXES) and code not in ('370', '371'):
        return 'Retail / General Commercial'
    if len(code) == 3 and code.startswith('9') and code >= '940':
        return 'Industrial'
    pt = str(row.get('PROPTYPE') or '').upper()
    return {
        'RESIDENTIAL': 'Other Residential', 'MULTI-FAMILY': 'Large Multi-Family (5+ units)',
        'COMMERCIAL': 'Other Commercial', 'INDUSTRIAL': 'Industrial',
        'AGRICULTURAL': 'Agricultural', 'MOBILE HOME': 'Other Residential',
    }.get(pt, 'Other')


CONDO_MAX_SQFT = 1_500                       # mapped parcel area; condo units are drawn as building-footprint slivers
gdf['_parcel_sqft'] = gdf.geometry.area      # CRS is in US feet (EPSG:6570)
gdf['PROPERTY_CATEGORY'] = gdf.apply(classify, axis=1)
_parking = gdf['PROPERTY_CATEGORY'] == 'Transportation - Parking'
gdf.loc[(gdf['BLDGVAL'] <= 0) & ~_parking, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print(gdf['PROPERTY_CATEGORY'].value_counts())

PROPERTY_CATEGORY
Single Family Residential         17182
Vacant Land                        4887
Condo & Townhome                   3003
Office / Commercial Condo          1224
Retail / General Commercial         920
Small Multi-Family (2-4 units)      444
Industrial                          182
Transportation - Parking            142
Large Multi-Family (5+ units)       141
Other Commercial                     85
Other                                73
Hotel                                30
Other Residential                    12
Name: count, dtype: int64


### 3c-2. Resolve the low-bill parcels: exempt, FILOT, or bill timing

A low 2026 bill can mean an exemption, a fee-in-lieu agreement, or simply timing: a building
finished, a parcel split, or a legal-residence application approved after the bill was computed.
Owner names come from the 2021 layer (Step 2d), so the owner test is approximate. A low bill
therefore means different things for different parcels:

| Rule | Treatment |
|---|---|
| Owner is government, church, school, university, hospital, housing authority or a known nonprofit, **and** billed < 50% of statutory | **Exempt** — held out |
| Non-residential category (office, retail, hotel, industrial, 5+ apartments, parking, other commercial, commercial vacant land), billed < 50% of statutory, owner not exempt | **FILOT / special arrangement** — held out (user decision) |
| Residential category (single family, condo, 2–4 units, residential lots), owner not exempt | **Kept** and modeled at its statutory value. The low bill reflects timing (new construction, sale, new plat), not a tax arrangement. FILOTs are not used for owner-occupied housing |
| `TAXMKTVAL = 0` | **No value** — held out |

In [7]:
# Step 3c-2: exempt / FILOT / timing
EXEMPT_OWNER_PATTERN = (
    r"CITY OF|\bCOUNTY\b|GREENVILLE CO\b|GVILLE CO\b|STATE OF|SOUTH CAROLINA DEPT|\bS ?C DEPT|UNITED STATES|\bUSA\b|"
    r"HOUSING AUTH|AUTHORITY|COMMISSION|SCHOOL|UNIVERSITY|COLLEGE|\bTECH\b|HOSPITAL|HEALTH SYS|BON SECOURS|PRISMA|"
    r"CHURCH|\bCH\b|BAPTIST|METHODIST|PRESBYTERIAN|CATHOLIC|EPISCOPAL|LUTHERAN|DIOCESE|PARISH|CONGREGATION|"
    r"MINISTR|TEMPLE|MOSQUE|SYNAGOGUE|WORSHIP|TABERNACLE|FELLOWSHIP|CEMETERY|"
    r"HABITAT|HOMES OF HOPE|GENESIS HOMES|UNITED HOUSING|HOUSING FUND|FOUNDATION|Y ?M ?C ?A|MUSEUM|ARENA DISTRICT|"
    r"COMMUNITY ECONOMIC|COMMUNITY DEV|\bINC\b.*\bTRS\b|TRUSTEES"
)
RESIDENTIAL_CATS = {'Single Family Residential', 'Condo & Townhome', 'Small Multi-Family (2-4 units)', 'Other Residential'}
_owner = gdf['OWNAM1'].fillna('').astype(str).str.upper()
gdf['exempt_owner'] = _owner.str.contains(EXEMPT_OWNER_PATTERN, regex=True)
_is_resid = gdf['PROPERTY_CATEGORY'].isin(RESIDENTIAL_CATS) | (
    (gdf['PROPERTY_CATEGORY'] == 'Vacant Land') & gdf['LANDUSE'].astype(str).str.startswith('11'))

gdf['no_value'] = (gdf['TAXMKTVAL'] <= 0).astype(int)
# City / county / housing-authority property is constitutionally exempt (SC Const. Art. X, §4);
# any TOTTAX on it is stormwater/sanitation fees, so it is exempt whatever the bill says.
gdf['gov_owner'] = _owner.str.contains(
    r'^CITY OF GREENVILLE|^GREENVILLE COUNTY\b|^COUNTY OF GREENVILLE|HOUSING AUTHORITY', regex=True)
gdf['exempt'] = (((gdf['low_bill'] == 1) & gdf['exempt_owner']) | gdf['gov_owner']).astype(int)
gdf['filot_or_special'] = ((gdf['low_bill'] == 1) & ~gdf['exempt_owner'] & ~_is_resid).astype(int)
gdf['bill_timing_mismatch'] = ((gdf['low_bill'] == 1) & ~gdf['exempt_owner'] & _is_resid).astype(int)
gdf['full_exmp'] = ((gdf['no_value'] == 1) | (gdf['exempt'] == 1) | (gdf['filot_or_special'] == 1)).astype(int)

_tot_mv = gdf['TAXMKTVAL'].sum()
for _flag, _label in [('no_value', 'No taxable value (new plats / common elements)'),
                      ('exempt', 'Exempt owner (incl. all city/county/housing-authority parcels)'),
                      ('filot_or_special', 'FILOT / special arrangement (non-residential)'),
                      ('bill_timing_mismatch', 'Residential low bill — KEPT (timing)')]:
    _m = gdf[_flag] == 1
    print(f"{_label:<50} {_m.sum():>6,} parcels  ${gdf.loc[_m, 'TAXMKTVAL'].sum() / 1e9:5.2f}B  ({gdf.loc[_m, 'TAXMKTVAL'].sum() / _tot_mv:.1%} of value)")

print("\nHeld out by category:")
print(gdf[gdf['full_exmp'] == 1].groupby('PROPERTY_CATEGORY').agg(
    n=('PIN', 'size'), market_value=('TAXMKTVAL', 'sum')).sort_values('market_value', ascending=False))
print("\nLargest FILOT / special-arrangement parcels:")
print(gdf[gdf['filot_or_special'] == 1].nlargest(12, 'TAXMKTVAL')[
    ['PIN', 'OWNAM1', 'PROPERTY_CATEGORY', 'TAXMKTVAL', 'TOTTAX', 'statutory_bill', 'bill_share']].round(2).to_string(index=False))
print("\nBill share of parcels kept (should cluster near 1.0):")
print(gdf.loc[gdf['full_exmp'] == 0, 'bill_share'].describe(percentiles=[.05, .25, .5, .75, .95]).round(3))

No taxable value (new plats / common elements)        164 parcels  $ 0.00B  (0.0% of value)
Exempt owner (incl. all city/county/housing-authority parcels)  1,091 parcels  $ 0.91B  (5.0% of value)
FILOT / special arrangement (non-residential)         397 parcels  $ 1.41B  (7.8% of value)
Residential low bill — KEPT (timing)                1,884 parcels  $ 0.71B  (3.9% of value)

Held out by category:
                                  n  market_value
PROPERTY_CATEGORY                                
Office / Commercial Condo       114     720909050
Large Multi-Family (5+ units)    51     583020480
Vacant Land                     991     566428810
Industrial                       46     129014920
Transportation - Parking         51      96346000
Retail / General Commercial      45      68778550
Other                            43      67531900
Single Family Residential       244      42222170
Other Commercial                 15      21664920
Small Multi-Family (2-4 units)   32      118841

### 3c-3. Refined categories: splitting the net-loser buckets

The first-pass buckets lumped together groups that behave very differently under an LVT. They are
split here so the results show *who* pays more and why. This changes labels only, not anyone's tax.

| From | Split into | Why it matters |
|---|---|---|
| Single Family Residential | **SFR — Owner-Occupied** (4% legal residence) / **SFR — Other** (6%: rentals, second homes, and owner-occupied homes without a legal-residence filing) | Different policy constituencies, and they already face different ratios |
| Vacant Land | Reported as one category, **Vacant & Undeveloped Land**. It combines **Vacant — Residential Lots** (individual lots and builder/developer subdivision inventory, plus code-1101 lots carrying only a shed, garage or outbuilding; deeded condo storage lockers on code 1101 go to **Condo & Townhome**), **Vacant — Commercial Land**, and **HOA, Rail/Utility & Development Land** (HOA common areas, railroad and utility rights-of-way, and construction sites or campus land with a built-use code but no building value yet) | Common areas, rights-of-way and sites under construction are not ordinary vacant land |
| Retail / General Commercial | **Auto-Oriented** (dealers, gas stations, car washes, auto service: codes 300–360), **Retail & Dining** (510–513 convenience and grocery, 520 general retail, 522 showroom, 523 drug store, 610 fast food, 620 full-service restaurant, 630 bar), **Shopping Centers & Plazas** (521 strip center, 545 strip retail, 550 neighborhood center), **Big Box & Mall Anchors** (530 discount, 531 discount warehouse, 560 mall, 561 anchor retail, 570 department store), **Retail / Other** | The auto-oriented and drive-thru uses are the most land-intensive commercial formats |
| Office | **Small Office (incl. converted houses)** (code 423) split from larger office | Small offices behave like land-heavy older houses |
| Industrial | **Warehouse / Light Industrial** (940/930) split from heavier industrial | |
| Other Commercial | **Golf / Country Club / Recreation Land** (750) | Large acreage, low building share |

In [8]:
# Step 3c-3: refined categories
_owner = gdf['OWNAM1'].fillna('').astype(str).str.upper()
_code = gdf['LANDUSE'].astype(str).str.strip()
_cat = gdf['PROPERTY_CATEGORY'].copy()
_vac = _cat == 'Vacant Land'
_util = _owner.str.contains(r'RAILWAY|RAILROAD|DUKE POWER|DUKE ENERGY|PIEDMONT NATURAL|BELLSOUTH|AT&T|TELEPHONE')
_hoa = (_owner.str.contains(r'OWNERS|HOMEOWNERS|\bHOA\b|ASSOCIATION|\bASSN\b|COMMUNITY ASSOC|MASTER ASSOC')
        | _code.isin(['1181', '205']))
_res_lot = _vac & _code.str.startswith('11')

_cat[_res_lot] = 'Vacant — Residential Lots'
# Code 1101 (residential, minor improvement): deeded condo storage lockers -> Condo & Townhome;
# the rest are lots with only a shed/garage/outbuilding -> residential lots
_minor = (_code == '1101') & (_cat == 'Other Residential')
_storage = _minor & gdf['DESCR'].fillna('').astype(str).str.upper().str.contains('STORAGE')
_cat[_storage] = 'Condo & Townhome'
_cat[_minor & ~_storage] = 'Vacant — Residential Lots'
_cat[_vac & (_code == '6800')] = 'Vacant — Commercial Land'
_cat[_vac & ~_code.str.startswith('11') & (_code != '6800')] = 'HOA, Rail/Utility & Development Land'
_cat[_hoa & (_vac | (_cat == 'Other'))] = 'HOA, Rail/Utility & Development Land'
_cat[_util] = 'HOA, Rail/Utility & Development Land'
_cat[_code.isin(['300', '301', '310', '330', '332', '360']) & ~_vac] = 'Auto-Oriented Commercial'
_cat[_code.isin(['610', '620', '630']) & ~_vac] = 'Retail & Dining'
# Retail codes per the county code list
_cat[_code.isin(['510', '511', '512', '513', '520', '522', '523']) & ~_vac] = 'Retail & Dining'   # convenience, grocery, general, showroom, drug store
_cat[_code.isin(['521', '545', '550']) & ~_vac] = 'Shopping Centers & Plazas'                     # strip centers, neighborhood centers
_cat[_code.isin(['530', '531', '560', '561', '570']) & ~_vac] = 'Big Box & Mall Anchors'          # discount, warehouse club, mall, anchors, dept stores
_cat[_cat == 'Retail / General Commercial'] = 'Retail / Other'
_cat[(_code == '750') & ~_vac] = 'Golf / Country Club / Recreation'
_cat[_code.isin(['940', '930']) & ~_vac] = 'Warehouse / Light Industrial'
_cat[(_code == '423') & ~_vac] = 'Small Office'
_sfr = _cat == 'Single Family Residential'
_cat[_sfr & (gdf['legal_residence'] == 1)] = 'SFR — Owner-Occupied'
_cat[_sfr & (gdf['legal_residence'] == 0)] = 'SFR — Other'
# One land category for everything with no (or only a token) building
_cat[_cat.isin(['Vacant — Residential Lots', 'Vacant — Commercial Land',
                'HOA, Rail/Utility & Development Land'])] = 'Vacant & Undeveloped Land'
gdf['PROPERTY_CATEGORY_BROAD'] = gdf['PROPERTY_CATEGORY'].replace({'Condo & Townhome': 'Condominium'})   # standard name for the cross-city CSV
gdf['PROPERTY_CATEGORY'] = _cat

print("Modeled parcels by refined category:")
print(gdf.loc[gdf['full_exmp'] == 0, 'PROPERTY_CATEGORY'].value_counts())
_resid = gdf[(gdf['full_exmp'] == 0) & gdf['PROPERTY_CATEGORY'].isin(['Other', 'Other Commercial', 'Other Residential'])]
print("\nCodes inside the residual buckets (taxable parcels):")
print(_resid.groupby(['PROPERTY_CATEGORY', 'LANDUSE']).size().sort_values(ascending=False).head(10))

Modeled parcels by refined category:
PROPERTY_CATEGORY
SFR — Owner-Occupied                13049
Vacant & Undeveloped Land            3923
SFR — Other                          3889
Condo & Townhome                     3002
Office / Commercial Condo             915
Retail & Dining                       593
Small Multi-Family (2-4 units)        412
Small Office                          195
Shopping Centers & Plazas              99
Auto-Oriented Commercial               97
Warehouse / Light Industrial           96
Transportation - Parking               91
Large Multi-Family (5+ units)          90
Retail / Other                         67
Other Commercial                       66
Industrial                             40
Hotel                                  29
Big Box & Mall Anchors                 19
Other                                  13
Golf / Country Club / Recreation        4
Other Residential                       1
Name: count, dtype: int64

Codes inside the residual buckets (t

### 3d. Validate against the observed bills (parcel-level ground truth)

The layer carries each parcel's actual 2026 bill, so the reconstruction is checked directly. For
taxable 6%-class parcels in district 500, the published 2026 full-stack rate (330.3 mills) times the
assessed value should reproduce the observed `TOTTAX` to within a few percent. The remainder is
stormwater and sanitation charges, partial-year changes, and rollback taxes.

In [9]:
# Step 3d: published-millage reconstruction of observed bills, district 500, 6% class, taxable
# (parcels whose bill does not match their attributes — Step 3c-2 — are excluded from this check)
_v = gdf[(gdf['DIST'] == '500') & (gdf['legal_residence'] == 0) & (gdf['full_exmp'] == 0)
         & (gdf['bill_timing_mismatch'] == 0)]
_recon = (_v['TAXMKTVAL'] * RATIO_OTHER * DISTRICT_500_TOTAL_MILLAGE / 1000).sum()
_obs = _v['TOTTAX'].sum()
BILL_GAP_PCT = (_recon / _obs - 1) * 100
print(f"District 500, 6% class: reconstructed ${_recon:,.0f} vs observed ${_obs:,.0f}  gap {BILL_GAP_PCT:+.2f}%")

# Legal residences: implied non-school-operating stack
_lr = gdf[(gdf['DIST'] == '500') & (gdf['legal_residence'] == 1) & (gdf['homestead_flag'] == 0)
          & (gdf['full_exmp'] == 0) & (gdf['bill_timing_mismatch'] == 0)]
_lr_mills = _lr['TOTTAX'].sum() / (_lr['TAXMKTVAL'] * RATIO_LEGAL_RESIDENCE).sum() * 1000
print(f"District 500 legal residences: implied {_lr_mills:.1f} mills (= {DISTRICT_500_TOTAL_MILLAGE} less school operating, "
      f"implying ~{DISTRICT_500_TOTAL_MILLAGE - _lr_mills:.0f} school-operating mills removed by Act 388)")

District 500, 6% class: reconstructed $111,889,357 vs observed $115,037,834  gap -2.74%


District 500 legal residences: implied 188.0 mills (= 330.3 less school operating, implying ~142 school-operating mills removed by Act 388)


## Section 4 — Current city tax

The city levy is rebuilt for every taxable parcel inside the boundary:

```
assessed_land        = LANDVAL × ratio
assessed_improvement = BLDGVAL × ratio
homestead (assessed) = min(HOMESTEAD, TAXMKTVAL) × ratio     (improvements first, then land)
current city tax     = (assessed_land + assessed_improvement − homestead) × 81.4 / 1000
```

Two versions are computed: `current_tax_hs50k` (the $50K permanent statute) and `current_tax`
(the $75K tax-year-2026 amount, the baseline). In South Carolina the state reimburses local governments for the homestead exemption
through the Homestead Exemption Fund. Raising it to $75K therefore lowers what homestead owners
pay **without** raising anyone else's city rate. The 81.4-mill rate is held fixed, and the LVT is
revenue-neutral against the taxpayer-paid city levy at $75K.

**Revenue control.** The city's FY2025–26 budget workshop put general-fund property taxes at about
$71.9M. That figure also covers vehicles, business personal property, FILOT fees and TIF increment,
so the modeled **real-property** levy should come in below it. The parcel-level check in 3d is the
primary validation.

In [10]:
# Step 4: current city tax
gdf['assessed_land'] = gdf['LANDVAL'] * gdf['assessment_ratio']
gdf['assessed_improvement'] = gdf['BLDGVAL'] * gdf['assessment_ratio']
gdf['millage_rate'] = CITY_MILLAGE


def homestead_assessed(amount: float) -> pd.Series:
    mv = np.where(gdf['homestead_flag'] == 1, np.minimum(amount, gdf['TAXMKTVAL']), 0.0)
    return pd.Series(mv, index=gdf.index) * gdf['assessment_ratio']


gdf['homestead_assessed_50k'] = homestead_assessed(HOMESTEAD_CURRENT)
gdf['homestead_assessed_75k'] = homestead_assessed(HOMESTEAD_PROPOSED)
gdf['homestead_assessed'] = homestead_assessed(HOMESTEAD_MODELED)

rev_50k, _, _tmp = calculate_current_tax(
    df=gdf, tax_value_col='TAXMKTVAL', millage_rate_col='millage_rate',
    land_value_col='assessed_land', improvement_value_col='assessed_improvement',
    exemption_col='homestead_assessed_50k', exemption_flag_col='full_exmp',
)
gdf['current_tax_hs50k'] = _tmp['current_tax']

current_revenue, _, gdf = calculate_current_tax(
    df=gdf, tax_value_col='TAXMKTVAL', millage_rate_col='millage_rate',
    land_value_col='assessed_land', improvement_value_col='assessed_improvement',
    exemption_col='homestead_assessed', exemption_flag_col='full_exmp',
)

BUDGET_FY2026_PROPERTY_TAXES = 71_900_000   # FY2025-26 budget workshop, all property types (proposal stage)
print(f"Modeled city real-property levy, $50K homestead (permanent statute): ${rev_50k:,.0f}")
print(f"Modeled city real-property levy, baseline ($75K homestead, TY2026):  ${current_revenue:,.0f}")
print(f"Homestead increase cost (state-reimbursed):                       ${rev_50k - current_revenue:,.0f}")
print(f"Taxable assessed value (pre-homestead): ${(gdf.loc[gdf['full_exmp'] == 0, 'assessed_land'] + gdf.loc[gdf['full_exmp'] == 0, 'assessed_improvement']).sum():,.0f}")
print(f"Share of FY2025-26 budgeted city property taxes (${BUDGET_FY2026_PROPERTY_TAXES / 1e6:.1f}M, all property types): "
      f"{current_revenue / BUDGET_FY2026_PROPERTY_TAXES:.1%}")
_in = gdf['in_city_tax_district'] & (gdf['full_exmp'] == 0)
print(f"  of which parcels in city tax districts: ${gdf.loc[_in, 'current_tax'].sum():,.0f}; "
      f"inside the boundary but outside a city district: ${gdf.loc[~gdf['in_city_tax_district'] & (gdf['full_exmp'] == 0), 'current_tax'].sum():,.0f}")

Modeled city real-property levy, $50K homestead (permanent statute): $64,331,962
Modeled city real-property levy, baseline ($75K homestead, TY2026):  $64,013,019
Homestead increase cost (state-reimbursed):                       $318,943
Taxable assessed value (pre-homestead): $798,291,962
Share of FY2025-26 budgeted city property taxes ($71.9M, all property types): 89.0%
  of which parcels in city tax districts: $64,013,019; inside the boundary but outside a city district: $0


## Section 5 — 4:1 split-rate with a flat $50K building exemption

The reform is solved on **assessed** values (ratio already applied), so the 4% vs 6% class
difference is untouched:

```
homestead (assessed)  allocated pro rata:  land_h = land − H·land/(land+bldg),  bldg_h = bldg − H·bldg/(land+bldg)
building exemption    taxable_bldg = max(0, bldg_h − $50,000 × ratio)      (never spills onto land)
taxable_land          = land_h
new tax               = taxable_land × land_mills + taxable_bldg × bldg_mills,   land_mills = 4 × bldg_mills
```

The $50K building exemption is the same dollar amount for every parcel, from a bungalow to a
tower. It removes a large share of a small home's building value and a negligible share of a
large building's.
Fully exempt, FILOT and no-value parcels are held out and excluded from the export.

In [11]:
# Step 5: split-rate + flat building exemption
n_exempt = int((gdf['full_exmp'] == 1).sum())
gdf = gdf[gdf['full_exmp'] == 0].copy()

_tot = (gdf['assessed_land'] + gdf['assessed_improvement']).replace(0, np.nan)
gdf['land_after_homestead'] = (gdf['assessed_land']
                               - gdf['homestead_assessed'] * gdf['assessed_land'] / _tot).fillna(0).clip(lower=0)
gdf['bldg_after_homestead'] = (gdf['assessed_improvement']
                               - gdf['homestead_assessed'] * gdf['assessed_improvement'] / _tot).fillna(0).clip(lower=0)
gdf['bldg_exemption_assessed'] = BUILDING_EXEMPTION * gdf['assessment_ratio']
gdf['reform_land'] = gdf['land_after_homestead']
gdf['reform_bldg'] = (gdf['bldg_after_homestead'] - gdf['bldg_exemption_assessed']).clip(lower=0)

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='reform_land',
    improvement_value_col='reform_bldg',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} exempt / FILOT / no-value parcels (excluded from model, export, and charts).")
print(f"Modeled parcels: {len(gdf):,}")
print(f"Current city millage: {CITY_MILLAGE:.2f}")
print(f"Land millage: {land_millage:.4f}   Improvement millage: {improvement_millage:.4f}")
print(f"Revenue check: ${new_revenue:,.0f} vs current ${gdf['current_tax'].sum():,.0f}")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print(f"Parcels whose whole building value is exempted: {(gdf['reform_bldg'] <= 0).sum() - (gdf['assessed_improvement'] <= 0).sum():,}")
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate + $50K Building Exemption (city levy)")

Held out 1,635 exempt / FILOT / no-value parcels (excluded from model, export, and charts).
Modeled parcels: 26,690
Current city millage: 81.40
Land millage: 187.7612   Improvement millage: 46.9403
Revenue check: $64,013,019 vs current $64,013,019
Parcels whose whole building value is exempted: 1,486

greenville_sc — 4:1 Split-Rate + $50K Building Exemption (city levy)


                        Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
            SFR — Owner-Occupied  13049         $71,852        0.3%         $6         $-49   -4.7%      -5.5%            20.9%            39.2%
       Vacant & Undeveloped Land   3923      $3,042,453      100.5%       $776         $173  119.3%     130.7%            93.3%             5.2%
                     SFR — Other   3889       $-168,589       -3.8%       $-43        $-119  -18.4%     -15.6%            14.0%            58.1%
                Condo & Townhome   3002       $-266,684       -6.8%       $-89        $-127  -14.6%     -14.2%            12.3%            58.0%
       Office / Commercial Condo    915       $-779,348       -9.2%      $-852        $-222   -3.9%     -10.4%            28.9%            50.2%
                 Retail & Dining    593        $126,062        3.8%       $213        $-144   -3.0%      -8.4%            32.5%   

In [12]:
# Step 5b: artifact scan (validate.md Gate 5)
_d = gdf.copy()
_d['bldg_share'] = _d['taxable_improvement_value'] / (
    _d['taxable_land_value'] + _d['taxable_improvement_value']).replace(0, np.nan)
scan = _d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('BLDGVAL', lambda s: (s <= 1000).mean()),
    current_tax=('current_tax', 'sum'),
    new_tax=('new_tax', 'sum'),
).sort_values('median_pct', ascending=False)
_ceiling = land_millage / CITY_MILLAGE - 1
print(f"Land-only ceiling: {_ceiling * 100:+.1f}%   building-only floor: {(improvement_millage / CITY_MILLAGE - 1) * 100:+.1f}%")
print(scan.round(3).to_string())

Land-only ceiling: +130.7%   building-only floor: -42.3%
                                      n  median_pct  median_bldg_share  pct_zero_bldg   current_tax       new_tax
PROPERTY_CATEGORY                                                                                                
Vacant & Undeveloped Land          3923     130.665              0.000          0.921  3.026318e+06  6.068771e+06
Transportation - Parking             91      74.424              0.000          0.440  2.090943e+05  2.857826e+05
Other Residential                     1      34.499              0.000          0.000  2.189500e+02  2.944850e+02
Other                                13      17.108              0.535          0.000  6.253195e+04  8.684922e+04
Small Office                        195      10.619              0.540          0.000  3.491852e+05  4.071370e+05
Retail / Other                       67       7.897              0.604          0.000  2.596323e+05  2.787523e+05
Auto-Oriented Commercial       

## Section 6 — Homestead: $50K today vs. $75K, and exploration charts

The **$50K→$75K** comparison under today's system (81.4 mills, no LVT) isolates the homestead
change from the LVT. The main LVT comparison above uses $75K in both the baseline and the reform.

In [13]:
# Step 6a: homestead change under the current system
hs = gdf[gdf['homestead_flag'] == 1].copy()
hs['hs_saving'] = hs['current_tax_hs50k'] - hs['current_tax']
print(f"Homestead parcels modeled: {len(hs):,}")
print(f"City tax on homestead parcels: ${hs['current_tax_hs50k'].sum():,.0f} at $50K -> ${hs['current_tax'].sum():,.0f} at $75K "
      f"(saving ${hs['hs_saving'].sum():,.0f}, reimbursed by the state)")
print(f"Median city-tax saving per homestead parcel: ${hs['hs_saving'].median():,.0f}  "
      f"(max possible = $25,000 x 4% x 81.4/1000 = ${25_000 * 0.04 * CITY_MILLAGE / 1000:,.0f})")
print(f"Homestead parcels paying $0 city tax: {(hs['current_tax_hs50k'] <= 0.5).sum():,} at $50K -> {(hs['current_tax'] <= 0.5).sum():,} at $75K")
print(f"Homestead parcels under the 4:1 LVT (vs $75K baseline): median change {hs.loc[hs['current_tax'] > 0, 'tax_change_pct'].median():+.1f}%, "
      f"total ${hs['current_tax'].sum():,.0f} -> ${hs['new_tax'].sum():,.0f}")
homestead_comparison = pd.DataFrame({
    'Scenario': ['$50K homestead (permanent law), 81.4 mills', 'Baseline: $75K homestead (2026), 81.4 mills', '4:1 LVT with $75K homestead'],
    'City tax on homestead parcels': [hs['current_tax_hs50k'].sum(), hs['current_tax'].sum(), hs['new_tax'].sum()],
    'Median per parcel': [hs['current_tax_hs50k'].median(), hs['current_tax'].median(), hs['new_tax'].median()],
}).round(0)
print(homestead_comparison.to_string(index=False))

Homestead parcels modeled: 3,995
City tax on homestead parcels: $5,679,412 at $50K -> $5,360,469 at $75K (saving $318,943, reimbursed by the state)
Median city-tax saving per homestead parcel: $81  (max possible = $25,000 x 4% x 81.4/1000 = $81)
Homestead parcels paying $0 city tax: 29 at $50K -> 147 at $75K
Homestead parcels under the 4:1 LVT (vs $75K baseline): median change -10.1%, total $5,360,469 -> $5,402,889
                                   Scenario  City tax on homestead parcels  Median per parcel
 $50K homestead (permanent law), 81.4 mills                      5679412.0             1043.0
Baseline: $75K homestead (2026), 81.4 mills                      5360469.0              962.0
                4:1 LVT with $75K homestead                      5402889.0              919.0


### 6e. Greenville charts

Five charts alongside the standard report, comparing the proposal with current law (81.4 mills on
land and buildings alike, with the $75K 2026 homestead; the senior chart also shows the $50K
permanent statute). They are saved to `analysis/reports/greenville_sc/` with a `gvl_` prefix, so
they also appear in the parcel map's chart gallery. Color: blue = pays less, red = pays more,
gray = about the same (±2%).

In [14]:
# Step 6e: custom charts (proposal vs current law)
from matplotlib.patches import Patch
_d = gdf.copy()
_R = _d['current_tax'].sum()
_sfr = _d['PROPERTY_CATEGORY_BROAD'] == 'Single Family Residential'
from matplotlib.ticker import FuncFormatter, PercentFormatter

REPORT_DIR = REPO_ROOT / 'analysis' / 'reports' / CITY_NAME
REPORT_DIR.mkdir(parents=True, exist_ok=True)
C_LESS, C_MORE, C_SAME = '#2a78d6', '#e34948', '#c3c2b7'
C_EMPH, C_MUTED = '#2a78d6', '#c3c2b7'
INK, INK2, GRID, SURF = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
plt.rcParams.update({
    'figure.facecolor': SURF, 'axes.facecolor': SURF, 'savefig.facecolor': SURF,
    'axes.edgecolor': GRID, 'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK,
    'text.color': INK, 'font.size': 10.5, 'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.spines.left': False,
    'text.parse_math': False,
})
SUBTITLE = 'Greenville, SC · city levy (81.4 mills) · 4:1 split-rate + $50K building exemption · revenue-neutral'


def _finish(fig, ax, title, subtitle=SUBTITLE, note=None, fname=None):
    from matplotlib.transforms import offset_copy
    fig.text(0.01, 1.0, title, ha='left', va='bottom', fontsize=15, fontweight='bold', color=INK,
             transform=offset_copy(fig.transFigure, fig=fig, y=26, units='points'))
    fig.text(0.01, 1.0, subtitle, ha='left', va='bottom', color=INK2, fontsize=10,
             transform=offset_copy(fig.transFigure, fig=fig, y=8, units='points'))
    if note:
        fig.text(0.01, 0.0, note, ha='left', va='top', color=INK2, fontsize=8.5,
                 transform=offset_copy(fig.transFigure, fig=fig, y=-6, units='points'))
    fig.savefig(REPORT_DIR / fname, dpi=200, bbox_inches='tight')
    plt.close(fig)


_pct_s = pd.Series(np.where(_d['current_tax'] > 0, (_d['new_tax'] - _d['current_tax']) / _d['current_tax'] * 100, np.nan),
                   index=_d.index)
_chg_s = _d['new_tax'] - _d['current_tax']
_cats = _d.assign(_pct=_pct_s, _chg=_chg_s).groupby('PROPERTY_CATEGORY').agg(
    n=('PIN', 'size'), median_pct=('_pct', 'median'), net=('_chg', 'sum'),
    less=('_pct', lambda v: (v < -2).mean()), more=('_pct', lambda v: (v > 2).mean()))
_cats = _cats[(_cats['n'] >= 20) | (_cats.index == 'Big Box & Mall Anchors')]

# Chart 1: median % change by refined category
_t = _cats.sort_values('median_pct')
fig, ax = plt.subplots(figsize=(10, 8.5))
_y = np.arange(len(_t))
ax.barh(_y, _t['median_pct'], height=0.62, color=[C_LESS if v < 0 else C_MORE for v in _t['median_pct']],
        edgecolor=SURF, linewidth=1)
ax.set_yticks(_y, [f"{c}  ({n:,})" for c, n in zip(_t.index, _t['n'])])
ax.axvline(0, color=INK2, lw=0.8)
for yi, v in zip(_y, _t['median_pct']):
    ax.text(v + (2 if v >= 0 else -2), yi, f"{v:+.0f}%", va='center', ha='left' if v >= 0 else 'right',
            fontsize=9.5, color=INK)
ax.xaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.grid(axis='x', color=GRID, lw=0.6); ax.set_axisbelow(True)
ax.set_xlim(_t['median_pct'].min() - 18, _t['median_pct'].max() + 22)
ax.set_xlabel('Median change in city property tax per parcel')
_finish(fig, ax, 'Land-heavy uses pay more; homes, condos and apartments pay less',
        note='Categories with 20+ parcels, plus Big Box & Mall Anchors; parcel counts in parentheses. Vacant and parking parcels sit at the land-only ceiling by design.',
        fname='gvl_1_category_median_change.png')

# Chart 2: single-family homes by value fifth, proposal vs current law
_qs = pd.qcut(_d.loc[_sfr, 'TAXMKTVAL'], 5)
_qlab = [f"Q{i + 1}\n${iv.left / 1000:,.0f}K–${iv.right / 1000:,.0f}K" for i, iv in enumerate(_qs.cat.categories)]
_qlab[0] = _qlab[0].replace(_qlab[0].split('$')[1].split('K')[0], '0')
_qs = _qs.cat.codes
_m_exp = _pct_s[_sfr].groupby(_qs).median()
_win = (_chg_s[_sfr] < -0.5).groupby(_qs).mean()
fig, ax = plt.subplots(figsize=(10, 5.6))
_x = np.arange(5)
ax.bar(_x, _m_exp.values, width=0.55, color=[C_LESS if v < 0 else C_MORE for v in _m_exp.values], edgecolor=SURF, linewidth=1)
for xi, v, w in zip(_x, _m_exp.values, _win.values):
    ax.text(xi, v - 0.8 if v < 0 else v + 0.8, f"{v:+.0f}%", ha='center', va='top' if v < 0 else 'bottom',
            fontsize=11, fontweight='bold', color=INK)
    ax.text(xi, 0.8, f"{w:.0%} pay less", ha='center', va='bottom', fontsize=9, color=INK2)
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xticks(_x, _qlab)
ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
ax.grid(axis='y', color=GRID, lw=0.6); ax.set_axisbelow(True)
ax.set_ylabel('Median change in city tax')
ax.set_xlabel('Single-family homes by taxable market value (fifths)')
ax.set_ylim(min(_m_exp.min() * 1.2, -5), 6)
_finish(fig, ax, 'The least valuable homes see the largest cuts',
        note='Single-family parcels (owner-occupied and rental), compared with 2026 law. Values are 2026 taxable market value (2025 reassessment).',
        fname='gvl_2_home_value_quintiles.png')

# Chart 3: who pays less / about the same / more, share of parcels by category
_t = _cats.sort_values('less')
_same = 1 - _t['less'] - _t['more']
fig, ax = plt.subplots(figsize=(10, 8.5))
_y = np.arange(len(_t))
ax.barh(_y, _t['less'], color=C_LESS, height=0.62, edgecolor=SURF, linewidth=2)
ax.barh(_y, _same, left=_t['less'], color=C_SAME, height=0.62, edgecolor=SURF, linewidth=2)
ax.barh(_y, _t['more'], left=_t['less'] + _same, color=C_MORE, height=0.62, edgecolor=SURF, linewidth=2)
for yi, l in zip(_y, _t['less']):
    if l >= 0.08:
        ax.text(l / 2, yi, f"{l:.0%}", ha='center', va='center', color='white', fontsize=9, fontweight='bold')
ax.set_yticks(_y, _t.index)
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.legend(handles=[Patch(color=C_LESS, label='Pays less (>2%)'), Patch(color=C_SAME, label='About the same (±2%)'),
                   Patch(color=C_MORE, label='Pays more (>2%)')],
          loc='lower center', bbox_to_anchor=(0.45, 1.0), ncol=3, frameon=False)
ax.set_xlabel('Share of parcels')
_finish(fig, ax, 'Most homes, condos and apartments pay less', fname='gvl_3_share_paying_less.png',
        note='Categories with 20+ parcels, plus Big Box & Mall Anchors. Sorted by share paying less.')

# Chart 4: where the money moves (net $ by category)
_t = _cats.sort_values('net')
fig, ax = plt.subplots(figsize=(10, 8.5))
_y = np.arange(len(_t))
ax.barh(_y, _t['net'] / 1e3, height=0.62, color=[C_LESS if v < 0 else C_MORE for v in _t['net']],
        edgecolor=SURF, linewidth=1)
ax.set_yticks(_y, _t.index)
ax.axvline(0, color=INK2, lw=0.8)
for yi, v in zip(_y, _t['net'] / 1e3):
    ax.text(v + (25 if v >= 0 else -25), yi, f"{'+' if v >= 0 else '−'}${abs(v):,.0f}K", va='center',
            ha='left' if v >= 0 else 'right', fontsize=9.5)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: ('−' if v < 0 else '') + (f"${abs(v) / 1000:,.1f}M" if abs(v) >= 1000 else f"${abs(v):,.0f}K")))
ax.grid(axis='x', color=GRID, lw=0.6); ax.set_axisbelow(True)
_lo, _hi = (_t['net'] / 1e3).min(), (_t['net'] / 1e3).max()
ax.set_xlim(_lo * 1.55, _hi * 1.35)
ax.set_xlabel('Net change in city tax collected from the category (per year)')
_gross = _chg_s.clip(lower=0).sum()
_finish(fig, ax, 'Where the money moves', note=f"${_gross / 1e6:.1f}M a year of city tax moves off building-heavy parcels and onto land-heavy ones "
        f"({_gross / _R:.0%} of the ${_R / 1e6:.1f}M levy). Categories with 20+ parcels, plus Big Box & Mall Anchors.", fname='gvl_4_where_money_moves.png')

(REPORT_DIR / 'gvl_5_policy_options.png').unlink(missing_ok=True)   # retired chart

# Chart 6: senior homestead owners, median city bill
_hs = _d['homestead_flag'] == 1
_bars = pd.Series({
    '$50K homestead\n(permanent law)': _d.loc[_hs, 'current_tax_hs50k'].median(),
    '$75K homestead\n(2026)': _d.loc[_hs, 'current_tax'].median(),
    'Proposal: 4:1 + $50K\nbuilding exemption': _d.loc[_hs, 'new_tax'].median(),
})
fig, ax = plt.subplots(figsize=(8, 5.4))
_x = np.arange(len(_bars))
ax.bar(_x, _bars.values, width=0.55, color=[C_MUTED, C_MUTED, C_EMPH], edgecolor=SURF, linewidth=1)
for xi, v in zip(_x, _bars.values):
    ax.text(xi, v + 8, f"${v:,.0f}", ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xticks(_x, _bars.index)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(axis='y', color=GRID, lw=0.6); ax.set_axisbelow(True)
ax.set_ylabel('Median annual city tax')
ax.set_ylim(0, _bars.max() * 1.18)
_finish(fig, ax, 'Senior homeowners: lower bills under the building exemption',
        subtitle=f"Greenville, SC · {_hs.sum():,} homestead-exempt (65+/disabled/blind) parcels · city levy only",
        fname='gvl_6_senior_homestead.png')
print(sorted(p.name for p in REPORT_DIR.glob('gvl_*.png')))

['gvl_1_category_median_change.png', 'gvl_2_home_value_quintiles.png', 'gvl_3_share_paying_less.png', 'gvl_4_where_money_moves.png', 'gvl_6_senior_homestead.png']


In [15]:
# Step 6b: quick category preview (not part of the standard report)
fig, ax = plt.subplots(figsize=(10, 6))
_m = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
_m.plot.barh(ax=ax, color=['#2E8B57' if v < 0 else '#B22222' for v in _m])
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Greenville, SC — Median City-Tax Change by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()

## Section 7 — Census join, standard export, report, parcel map

In [16]:
# Census join — OPTIONAL, must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
_census_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct']
if not os.getenv('CENSUS_API_KEY'):
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in _census_cols:
        gdf[_col] = float('nan')
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=90)
                gdf = match_to_census_blockgroups(gdf, census_gdf)
                # census_gdf already carries demographics — no second merge (avoids _x/_y duplicates)
                if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                    gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
                if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                    gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
                print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
            except concurrent.futures.TimeoutError:
                print("Census API timed out — skipping census join")
                for _col in _census_cols:
                    gdf[_col] = float('nan')
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in _census_cols:
            gdf[_col] = float('nan')

Census join: 100.0% matched


In [17]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY_BROAD',   # standard names: keeps cross-city + equity charts correct
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export — GeoParquet + self-contained interactive HTML map
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
    property_category_col='PROPERTY_CATEGORY',          # refined categories in the map popups
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

  ✓ greenville_sc: 26,690 rows → ../../analysis/data/greenville_sc.csv  [model: split_rate_4to1_bldg_exempt_50k]
create_city_report: excluded 101 fully-exempt parcels (26,690 → 26,589 modeled).


  [warn] greenville_sc: non-standard property categories (will be preserved): ['Auto-Oriented Commercial', 'Big Box & Mall Anchors', 'Condo & Townhome', 'Golf / Country Club / Recreation', 'Retail & Dining', 'Retail / Other', 'SFR — Other', 'SFR — Owner-Occupied', 'Shopping Centers & Plazas', 'Small Office', 'Vacant & Undeveloped Land', 'Warehouse / Light Industrial']


  ✓ greenville_sc: 26,690 parcels → ../../analysis/maps/greenville_sc.parquet  [map export; 0 with record links]


  ✓ greenville_sc: interactive map → ../../analysis/reports/greenville_sc/parcel_map.html  [26,690 parcels]
Done.


## Validation summary and limitations

| Check | Result |
|---|---|
| Data | County GIS export of 2026-11-23: **tax-year 2026 bills on 2025-reassessment values**. 28,325 parcels inside the official city boundary, all in city tax districts 500–517 |
| Revenue match (Gate 1) | Published 2026 district-500 millage (330.3) × modeled 6% assessed value reproduces observed bills at **−2.7%** (remainder = stormwater/sanitation fees and partial-year changes). Legal residences imply ~188 mills (= 330.3 less Act 388 school operating). Modeled city real-property levy **$64.0M** at the $75K homestead ($64.3M at $50K) = 89% of the city's FY2025–26 budgeted property taxes ($71.9M, all property types) |
| Parcel count | 28,325 → **26,690 modeled**. Held out: 1,091 exempt-owner (incl. every city/county/housing-authority parcel), 397 FILOT/special (non-residential, incl. 51 large apartment complexes paying ~2% of their statutory bill), 164 no-value |
| Distribution (Gate 2) | Vacant & undeveloped land **+131%** (land-only ceiling), parking +74%, small office +11%, retail/other +8%, auto-oriented +3%; SFR owner-occupied **−5.5%**, SFR other −15.6%, condo & townhome −14%, retail & dining −8%, shopping centers −9%, big box & mall anchors −9%, office −10%, 2–4 units −16%, warehouses −18%, 5+ apartments −23%, hotels −27% (medians). Revenue-neutral at $64.0M |
| Census coverage (Gate 3) | 100% of parcels joined to 2022 block groups |
| PNGs (Gate 4) | 7 of 7 standard + 6 Greenville charts, `parcel_map.html` and `analysis/maps/greenville_sc.parquet` (no parcel links or owner names) |
| Artifact scan (Gate 5) | **Cap booking fixed.** The county books full land value and charges the whole 15% cap discount to the building, which on 72% of parcels inflated land shares (single-family capped: 40% land as booked vs 26% of market value); fixed by the proportional re-split in Section 2. A first version of that fix invented building value on capped vacant lots; corrected by leaving zero-building parcels all land. **Bill classes**: 27% of bills match no candidate, mostly recent sales billed on the earlier capped value; resolved by bill level (Section 3a). After both fixes building shares are plausible (single-family 67%, retail & dining 73%, office 75%, apartments 87%) and only vacant land and parking sit at the ceiling |

**The exported policy: 4:1 split + flat $50K building exemption.** Revenue-neutral on the city's
$64.0M levy, with land at 187.8 mills and buildings at 46.9 mills above the first $50,000:

- Single-family median **−7.6%**, with **65%** of homes paying less. The least valuable fifth
  (under $148K taxable) goes to **−37%**. Condos and townhomes: −14%.
- 65+ homestead owners: median **−10%** against the $75K baseline. Their median city bill is
  **$919**, against $962 under the 2026 $75K homestead and $1,043 under the $50K permanent statute.
- Residential neighborhoods: −21% in the lowest-income fifth of block groups, roughly flat in the
  highest; −20% in the fifth with the highest share of residents of color, −3.5% in the lowest.
- Land's share of the city levy rises from 27% today to **62%**.
- About $7.6M a year (12% of the levy) moves onto land-heavy parcels.

The 65+ homestead exemption is allocated **pro rata** between land and building.

**Limitations**

- **Identity fields are from 2021.** The 2026 export has no parcel number, owner, ratio code or
  homestead flag. Assessment class and homestead come from the 2026 bills; parcel number, owner and
  legal description are carried over by location from the 2021 layer and used only to recognize
  government, nonprofit, HOA and utility owners. Owners who bought or sold since 2021 can be
  misclassified in that test. Condo units are identified by parcel size (under 1,500 sq ft), which
  also captures some townhouses.
- **Exempt/FILOT detection is inferred** from bills and owner names. A large share of the held-out
  value is apartment complexes paying about 2% of their statutory bill, consistent with
  affordable-housing exemptions or fee agreements; the model cannot tell which.
- **Recent sales (about 1,900 residential parcels)** carry post-sale values with bills computed on
  the earlier capped value. They are modeled at the values in the export.
- **Cap re-split.** The proportional split assumes the cap discount applies to land and building
  alike. Where the cap exceeded the building value the county booked the parcel as all land, and it
  stays all land here.
- **Personal property, vehicles, and DOR-assessed manufacturing/utility property** (10.5%) are not
  in the parcel layer and are not shifted.
- **Legality.** South Carolina's constitution (Art. X, §1) fixes the assessment ratios and requires
  uniform millage within a class. A city split-rate would need state constitutional or statutory
  change; this is an impact model, not a legal pathway.
- **$75K homestead** is the tax-year-2026 amount under a one-year budget proviso for owners already
  eligible in 2025 (new 2026 eligibles get $50K, not separated here). S.768, pending in House Ways &
  Means, would change it permanently. The state reimburses localities, so the city millage is held fixed.